# FormulaLM — Evaluate QLoRA v2 (Validation-First + Informed Sets + Blind `test_real_v2`)

1. **Validation decision first (`val.jsonl`, 500 examples):** evaluates both frozen v1 and v2 on `val.jsonl` so model selection is strictly validation-driven.
2. **Original test sets (labeled `'informed by v1 error analysis'`):** evaluates v2 on `test_synthetic.jsonl` (500) and `test_real_DRAFT.csv` (80).
3. **Blind fresh set (`data/test_real_v2/test_real_v2.csv`, ~50 requests):** if present, evaluates Base 0-shot, Frozen v1, and v2 separately on the unseen blind set.

Frozen v1 (`results/ft_v1_*` and git tag `v1-frozen`) is never modified.

In [ ]:
# 1. Verify CUDA T4 GPU and mount Google Drive
import pathlib, torch

assert torch.cuda.is_available(), 'No CUDA GPU detected! Switch kernel to T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

In [ ]:
# 2. Clone / pull repo, install deps, ensure val.jsonl and test_synthetic.jsonl exist
import getpass, os, pathlib, subprocess, sys

REPO_DIR = pathlib.Path('/content/FormulaLM')
if not REPO_DIR.exists():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass.getpass('GitHub PAT for abubakarp789/FormulaLM (hidden, empty if public): ').strip()
    url = f'https://{token}@github.com/abubakarp789/FormulaLM.git' if token else 'https://github.com/abubakarp789/FormulaLM.git'
    subprocess.run(['git', 'clone', url, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd /content/FormulaLM
if '/content/FormulaLM/src' not in sys.path:
    sys.path.insert(0, '/content/FormulaLM/src')

%pip install -q formulas openpyxl pandas pyyaml requests tqdm accelerate peft "transformers>=4.51.0" "huggingface_hub>=0.30.0"

if not pathlib.Path('data/processed/test_synthetic.jsonl').is_file():
    !python -m src.data.build_dataset --version v2 --workers 2

In [ ]:
# 3. Evaluate QLoRA v2 on val (model selection) + original test sets (informed by v1 error analysis)
import pathlib
from src.eval.run_eval import TransformersBackend, evaluate

MODEL_ID = 'Qwen/Qwen3-1.7B'
REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'

V2_CANDIDATES = [
    '/content/drive/MyDrive/FormulaLM/qlora_v2/adapter',
    'outputs/qlora_v2/adapter',
]
V2_ADAPTER = next((p for p in V2_CANDIDATES if pathlib.Path(p).is_dir()), 'abubakar56/formulalm-qwen3-1.7b-lora-v2')
print('Loading v2 adapter from:', V2_ADAPTER, flush=True)

v2_backend = TransformersBackend(
    MODEL_ID, adapter_path=V2_ADAPTER, revision=REVISION, load_in_4bit=False, enable_thinking=False
)

eval_jobs = [
    ('ft_v2_val', 'data/processed/val.jsonl'),
    ('ft_v2_real', 'data/test_real/test_real_DRAFT.csv'),
    ('ft_v2_synthetic', 'data/processed/test_synthetic.jsonl'),
]
BLIND_CSV = pathlib.Path('data/test_real_v2/test_real_v2.csv')
if BLIND_CSV.is_file():
    eval_jobs.append(('ft_v2_real_v2_blind', str(BLIND_CSV)))

for run_name, dataset in eval_jobs:
    print(f'=== Running {run_name} ===', flush=True)
    m = evaluate(
        backend=v2_backend,
        dataset_path=dataset,
        run_name=run_name,
        seed=42,
        max_tokens=96,
        few_shot=0,
        batch_size=8,
        resume=True,
    )
    ov = m['overall']
    print(f"{run_name}: n={ov['n']} | syntax={100*ov['syntax_valid_rate']:.1f}% | exec_acc={100*ov['execution_accuracy']:.1f}% | exact={100*ov['exact_match_rate']:.1f}%", flush=True)

In [ ]:
# 4. Optional: also evaluate Base 0-shot and Frozen v1 on val.jsonl and blind test_real_v2.csv (if present)
import gc, torch
del v2_backend
gc.collect()
torch.cuda.empty_cache()

V1_CANDIDATES = [
    '/content/drive/MyDrive/FormulaLM/qlora_v1/adapter',
    'outputs/qlora_v1/adapter',
]
V1_ADAPTER = next((p for p in V1_CANDIDATES if pathlib.Path(p).is_dir()), 'abubakar56/formulalm-qwen3-1.7b-lora')

v1_backend = TransformersBackend(
    MODEL_ID, adapter_path=V1_ADAPTER, revision=REVISION, load_in_4bit=False, enable_thinking=False
)
for run_name, dataset in [('ft_v1_val', 'data/processed/val.jsonl')] + (
    [('ft_v1_real_v2_blind', str(BLIND_CSV))] if BLIND_CSV.is_file() else []
):
    m = evaluate(backend=v1_backend, dataset_path=dataset, run_name=run_name, seed=42, max_tokens=96, few_shot=0, batch_size=8, resume=True)
    ov = m['overall']
    print(f"{run_name}: n={ov['n']} | exec_acc={100*ov['execution_accuracy']:.1f}%", flush=True)

if BLIND_CSV.is_file():
    del v1_backend
    gc.collect()
    torch.cuda.empty_cache()
    base_backend = TransformersBackend(MODEL_ID, revision=REVISION, load_in_4bit=False, enable_thinking=False)
    m = evaluate(backend=base_backend, dataset_path=str(BLIND_CSV), run_name='baseline_zeroshot_real_v2_blind', seed=42, max_tokens=96, few_shot=0, batch_size=8, resume=True)
    print(f"baseline_zeroshot_real_v2_blind: n={m['overall']['n']} | exec_acc={100*m['overall']['execution_accuracy']:.1f}%")

In [ ]:
# 5. Produce comparison report (explicitly labeling v2 on v1 test sets as 'informed by v1 error analysis')
!python -m src.eval.compare_and_cluster \
  --ft-synthetic results/ft_v2_synthetic \
  --ft-real results/ft_v2_real \
  --ft-label "Fine-tuned v2 (informed by v1 error analysis)" \
  --examples-per-cluster 5 \
  --out results/ft_v2_comparison.md \
  --json-out results/ft_v2_comparison.json

In [ ]:
# 6. Push lightweight v2 evaluation results back to GitHub
PUSH_TO_GITHUB = True
if PUSH_TO_GITHUB:
    !git config user.name "abubakarp789"
    !git config user.email "abubakarp789@gmail.com"
    !git add results/ft_v2_* results/ft_v1_val results/ft_v1_real_v2_blind results/baseline_zeroshot_real_v2_blind 2>/dev/null || git add results/ft_v2_*
    !git commit -m "results: add QLoRA v2 evaluation results (validation + informed sets)"
    !git pull --rebase origin main
    !git push origin main